# µP Experiments: LR Sweep + Scaling Study

SVG Scaling Laws Project — Part 3

**Runtime**: A100 GPU

**Sections**:
1. Setup
2. µP LR Sweep (Tiny, 7 LRs) — ~35 min
3. µP Scaling Study (all 5 models) — ~2.5 hours

**Total estimated time**: ~3 hours

## 1. Setup

In [28]:
import os
base_dir = '/content/svg-scaling-project'

# train.py が存在するか
print(os.path.exists(f'{base_dir}/src/train.py'))

# mup_lr_sweep ディレクトリの中身
sweep_dir = f'{base_dir}/results/runs/mup_lr_sweep'
if os.path.exists(sweep_dir):
    for d in os.listdir(sweep_dir):
        sub = f'{sweep_dir}/{d}'
        files = os.listdir(sub) if os.path.isdir(sub) else []
        print(f'  {d}: {files}')
else:
    print(f'{sweep_dir} does not exist')

!cd /content/svg-scaling-project && python src/train.py \
    --config configs/tiny.yaml \
    --max-steps 10 \
    --mup \
    --device cuda


True
/content/svg-scaling-project/results/runs/mup_lr_sweep does not exist
Using device: cuda
Loading tokenized data...
Train tokens: 110,325,825, Val tokens: 11,195,284
Model parameters: 1,311,872 (1.31M) [µP]
  µP width_mult=1.000, base_width=128
Max steps: 10, Warmup steps: 0
Tokens per step: 16,384
Output dir: results/runs/tiny_20260426_041007
  param_group[0]: lr=3.00e-04, lr_scale=1.0000, n_params=655,360
  param_group[1]: lr=3.00e-04, lr_scale=1.0000, n_params=786,432
  param_group[2]: lr=3.00e-04, lr_scale=1.0000, n_params=1,152
step      0/10 | loss 8.3258 | lr 3.00e-04 | 0.3s | 16,384 tokens
  → Final eval is new best val_loss=7.7523, saved checkpoint

--- Training complete ---
  Total time: 1.2s
  Final train_loss: 7.5635
  Final val_loss: 7.7523
  Final val_ppl: 2326.83
  Best val_loss: 7.7523

All outputs saved to results/runs/tiny_20260426_041007


In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
!unzip -qo /content/drive/MyDrive/SVGScalingLawProject/svg-scaling-project.zip -d /content/svg-scaling-project


In [ ]:
!pip install -q pyyaml mup

In [32]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!ls /content/svg-scaling-project/data/tokenized/

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB
test.bin  tokenize_stats.json  train.bin  val.bin


## 2. µP LR Sweep (Tiny)

7 learning rates on log scale, including higher LRs that µP should stabilize.

Estimated time: ~35 min (7 × 5 min per Tiny 1-epoch)

In [33]:
import subprocess, time, json

learning_rates = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3', '1e-2', '3e-2']
base_dir = '/content/svg-scaling-project'
sweep_results = []

for lr in learning_rates:
    output_dir = f'{base_dir}/results/runs/mup_lr_sweep/lr_{lr}'
    print(f'\n{"=" * 60}')
    print(f'µP LR Sweep: Tiny, LR={lr}')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', f'{base_dir}/src/train.py',
        '--config', f'{base_dir}/configs/tiny.yaml',
        '--learning-rate', lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
        '--mup',
    ], cwd=base_dir)

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n→ LR={lr}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        sweep_results.append({'lr': lr, **s})
        print(f'  best_val_loss={s["best_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

print(f'\n{"=" * 60}')
print('µP LR Sweep Complete!')
print(f'{"=" * 60}')
for r in sweep_results:
    print(f'  LR={r["lr"]:>5s}: best_val_loss={r["best_val_loss"]:.4f}, '
          f'ppl={r["final_val_ppl"]:.2f}')


µP LR Sweep: Tiny, LR=3e-5

→ LR=3e-5: OK, 2.7 min
  best_val_loss=4.4752, final_ppl=91.12

µP LR Sweep: Tiny, LR=1e-4

→ LR=1e-4: OK, 2.7 min
  best_val_loss=4.0793, final_ppl=59.61

µP LR Sweep: Tiny, LR=3e-4

→ LR=3e-4: OK, 2.7 min
  best_val_loss=3.4554, final_ppl=31.67

µP LR Sweep: Tiny, LR=1e-3

→ LR=1e-3: OK, 2.7 min
  best_val_loss=3.2326, final_ppl=25.34

µP LR Sweep: Tiny, LR=3e-3

→ LR=3e-3: OK, 2.7 min
  best_val_loss=3.1445, final_ppl=24.68

µP LR Sweep: Tiny, LR=1e-2

→ LR=1e-2: OK, 2.7 min
  best_val_loss=3.1840, final_ppl=25.00

µP LR Sweep: Tiny, LR=3e-2

→ LR=3e-2: OK, 2.7 min
  best_val_loss=3.0449, final_ppl=21.67

µP LR Sweep Complete!
  LR= 3e-5: best_val_loss=4.4752, ppl=91.12
  LR= 1e-4: best_val_loss=4.0793, ppl=59.61
  LR= 3e-4: best_val_loss=3.4554, ppl=31.67
  LR= 1e-3: best_val_loss=3.2326, ppl=25.34
  LR= 3e-3: best_val_loss=3.1445, ppl=24.68
  LR= 1e-2: best_val_loss=3.1840, ppl=25.00
  LR= 3e-2: best_val_loss=3.0449, ppl=21.67


In [ ]:
# Determine optimal LR from sweep
best = min(sweep_results, key=lambda x: x['best_val_loss'])
optimal_lr = best['lr']
print(f'\nOptimal µP LR: {optimal_lr} (best_val_loss={best["best_val_loss"]:.4f})')
print('\nThis LR will be used for the scaling study below.')

## 3. µP Scaling Study (All 5 Models)

Train all 5 models with the optimal µP LR found above.

Estimated time: ~2.5 hours (Tiny 5m + Small 10m + Medium 20m + Large 30m + XL 80m)

In [35]:
models = ['tiny', 'small', 'medium', 'large', 'xl']
scaling_results = []

for name in models:
    output_dir = f'{base_dir}/results/runs/mup_scaling_study/{name}'
    print(f'\n{"=" * 60}')
    print(f'µP Scaling Study: {name} (LR={optimal_lr})')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', f'{base_dir}/src/train.py',
        '--config', f'{base_dir}/configs/{name}.yaml',
        '--learning-rate', optimal_lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
        '--mup',
    ], cwd=base_dir)

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n→ {name}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        scaling_results.append(s)
        print(f'  best_val_loss={s["best_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

print(f'\n{"=" * 60}')
print('µP Scaling Study Complete!')
print(f'{"=" * 60}')
for r in scaling_results:
    print(f'  {r["config_name"]:>8s}: {r["n_params"]/1e6:.1f}M, '
          f'best_val_loss={r["best_val_loss"]:.4f}, ppl={r["final_val_ppl"]:.2f}')


µP Scaling Study: tiny (LR=3e-2)

→ tiny: OK, 2.7 min
  best_val_loss=3.1148, final_ppl=23.49

µP Scaling Study: small (LR=3e-2)

→ small: OK, 5.6 min
  best_val_loss=2.6091, final_ppl=13.97

µP Scaling Study: medium (LR=3e-2)

→ medium: OK, 12.7 min
  best_val_loss=2.5767, final_ppl=13.90

µP Scaling Study: large (LR=3e-2)

→ large: OK, 31.2 min
  best_val_loss=2.1099, final_ppl=8.25

µP Scaling Study: xl (LR=3e-2)

→ xl: OK, 77.2 min
  best_val_loss=1.8623, final_ppl=7.23

µP Scaling Study Complete!
      tiny: 1.3M, best_val_loss=3.1148, ppl=23.49
     small: 3.4M, best_val_loss=2.6091, ppl=13.97
    medium: 12.2M, best_val_loss=2.5767, ppl=13.90
     large: 33.6M, best_val_loss=2.1099, ppl=8.25
        xl: 88.1M, best_val_loss=1.8623, ppl=7.23


## 4. Save Results to Drive

In [36]:
import shutil, os

# Save LR sweep results
drive_sweep = '/content/drive/MyDrive/SVGScalingLawProject/mup_lr_sweep_results'
os.makedirs(drive_sweep, exist_ok=True)
for lr in learning_rates:
    src_dir = f'{base_dir}/results/runs/mup_lr_sweep/lr_{lr}'
    dst_dir = f'{drive_sweep}/lr_{lr}'
    os.makedirs(dst_dir, exist_ok=True)
    for fname in ['summary.json', 'training_log.json', 'best_model.pt']:
        src = f'{src_dir}/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{dst_dir}/{fname}')
            print(f'Copied lr_{lr}/{fname}')

# Save scaling study results
drive_scaling = '/content/drive/MyDrive/SVGScalingLawProject/mup_scaling_study_results'
os.makedirs(drive_scaling, exist_ok=True)
for name in models:
    src_dir = f'{base_dir}/results/runs/mup_scaling_study/{name}'
    dst_dir = f'{drive_scaling}/{name}'
    os.makedirs(dst_dir, exist_ok=True)
    for fname in ['summary.json', 'training_log.json', 'best_model.pt']:
        src = f'{src_dir}/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{dst_dir}/{fname}')
            print(f'Copied {name}/{fname}')

print('\nAll µP results saved to Drive.')

Copied lr_3e-5/summary.json
Copied lr_3e-5/training_log.json
Copied lr_3e-5/best_model.pt
Copied lr_1e-4/summary.json
Copied lr_1e-4/training_log.json
Copied lr_1e-4/best_model.pt
Copied lr_3e-4/summary.json
Copied lr_3e-4/training_log.json
Copied lr_3e-4/best_model.pt
Copied lr_1e-3/summary.json
Copied lr_1e-3/training_log.json
Copied lr_1e-3/best_model.pt
Copied lr_3e-3/summary.json
Copied lr_3e-3/training_log.json
Copied lr_3e-3/best_model.pt
Copied lr_1e-2/summary.json
Copied lr_1e-2/training_log.json
Copied lr_1e-2/best_model.pt
Copied lr_3e-2/summary.json
Copied lr_3e-2/training_log.json
Copied lr_3e-2/best_model.pt
Copied tiny/summary.json
Copied tiny/training_log.json
Copied tiny/best_model.pt
Copied small/summary.json
Copied small/training_log.json
Copied small/best_model.pt
Copied medium/summary.json
Copied medium/training_log.json
Copied medium/best_model.pt
Copied large/summary.json
Copied large/training_log.json
Copied large/best_model.pt
Copied xl/summary.json
Copied xl/t